# arcroad · Remote verification v05 (2 Nov 2026)

**New in v05**
1. Secrets are read in cell 1, while you are at the screen: a locked screen can no longer time out the Mapillary key mid-run (the v04 failure).
2. Impossible photo-run passes are rejected: driven shorter than the straight line, no heading reversal (no U-turn) between the A and B ends, or a match radius wider than half the A–B gap. These made v04's 25 false "SHORTER" results.
3. Evidence E: U-turn and no-U-turn **traffic signs** detected in Mapillary imagery near each place.
4. Evidence F: **turn restrictions** mapped in OpenStreetMap (no_u_turn, only_straight_on) at each place.
5. arcroad's **own drives** (data/field/*-results.json) are read in, so places driven by arcroad are graded "Driven by arcroad" with the same rules as the website.
6. **Road class** per place (expressway mainline such as the ORR vs city road): detours on access-controlled roads are by design and are shown separately.
7. KartaView photo runs as an optional second drive source (off by default; turned on only after its terms are checked).

Checks arcroad's modelled detours **without anyone driving**, against independent evidence:

| Evidence | What it is | Licence | Independent of the model? |
|---|---|---|---|
| A. Model route | OpenStreetMap road graph, shortest legal drive A→B, separately for 2-wheelers, cars and heavy vehicles | ODbL | no (it *is* the model) |
| B. Driven traces | Public GPS traces uploaded to OpenStreetMap by people who actually drove past | ODbL, public trackpoints API | **yes: real drives** |
| C. Imagery detections | Microsoft ML road detections from aerial imagery | ODbL | **yes: from imagery** |
| B2. Timed drives with photos | Mapillary street-level sequences (photo every 1-2 s, capture time) | CC BY-SA 4.0 | **yes: real, timed drives you can look at** |
| C2. Other map | Overture Maps road segments not sourced from OSM | per-source (Overture) | **yes, where the source is not OSM** |
| D. TomTom route *(off by default)* | A commercial map's own route | TomTom terms (read before enabling) | yes |

**Zero Google Maps data** (AGENTS.md hard constraint). Colab compute is fine; Google map content is not.

### Rule 5, written before running (31 Oct 2026)
- **Hypothesis:** the model's network distance is what a legal driver must travel from side A to side B.
- **Competing explanation:** OpenStreetMap is missing a legal connection (a median opening, U-turn or slip road), so the real drive is far shorter.
- **What separates them:** a real drive from A to B (Evidence B). Near the model value supports the hypothesis; well under it supports the competing explanation. Imagery (C) and TomTom (D) can only *dispute*, never confirm.
- **Control:** generated pairs of points along ordinary two-way roads near each flagged stop (no detour expected; seeded). If Evidence B cannot reproduce MATCH on controls, it has no power and its results are not used.
- **Gate:** `pipeline.groundtruth` thresholds, unchanged (MATCH within max(150 m, 25%); SHORTER below 50%). Trace density gate for untimed public traces, fixed here: no gap over **60 m** between consecutive points inside a pass.

Privacy: trace coordinates are used in memory only and never written anywhere. Outputs are per-stop numbers.


### v04 changes (1 Nov 2026, before this run)
- **B-15 fix** (repo PR #15): a pass must come within 10 m of both A and B, and every sample counted as "at B" must sit on side B's carriageway in the road graph. The v02 run produced 25 impossible "shorter" passes; this removes them and cannot create new SHORTER verdicts.
- Overture and Microsoft layers read geometry back with `ST_AsWKB`.
- Saves each modelled car route as a line (`routes_*.geojson`) for the website's animated detour.
- Live progress file in Drive every 20 s.


In [ ]:
# 1. Setup (about 2 minutes)
!git clone -q https://github.com/rrmlogs8896/arcroad.git /content/arcroad 2>/dev/null || (cd /content/arcroad && git pull -q)
!pip -q install osmnx==2.* duckdb shapely folium
import sys, json, math, time, subprocess, datetime as dt, io
sys.path.insert(0, '/content/arcroad')
from pipeline import groundtruth as gt
from pipeline.geometry import haversine_m
COMMIT = subprocess.run(['git','-C','/content/arcroad','rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
RUN_DATE = dt.date.today().isoformat()
print('arcroad commit', COMMIT, '| run', RUN_DATE)
# Secrets, read now (v05): a locked screen later cannot block them.
def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
MAPILLARY_TOKEN = _secret('MAPILLARY_TOKEN')
TOMTOM_KEY = _secret('TOMTOM_KEY')
print('Mapillary key:', 'found' if MAPILLARY_TOKEN else 'missing, so Mapillary evidence is off', '| TomTom key:', 'found' if TOMTOM_KEY else 'not set')
# Does the Mapillary key actually work? One tiny request (a 100 m box in Hyderabad). The key itself is never printed.
if MAPILLARY_TOKEN:
    import requests as _rq
    try:
        _r = _rq.get('https://graph.mapillary.com/images?fields=id&bbox=78.3890,17.4765,78.3900,17.4775&limit=1',
                     headers={'Authorization': f'OAuth {MAPILLARY_TOKEN}'}, timeout=30)
        print('Mapillary API check:', {200: 'OK, the key works', 401: 'REJECTED (401): the key is wrong or expired; make a new client token at mapillary.com/dashboard/developers', 403: 'REFUSED (403): the app lacks read permission'}.get(_r.status_code, f'HTTP {_r.status_code}'))
    except Exception as _e:
        print('Mapillary API check: could not reach Mapillary (', type(_e).__name__, ')')


In [ ]:
# 2. Output folder in your Drive (Anthropic / Hyderabad Mobility Atlas / verify)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Anthropic/Hyderabad Mobility Atlas/verify'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
# 2b. Live progress file (so Claude can check from your phone)
# Every 20 s writes progress.json to the same Drive folder: which step is running, how long, what finished, any error.
# Contains no keys, no coordinates, no trace data.
import threading, json as _json, time as _time, datetime as _dt
PROG = {'run_started': _dt.datetime.now().isoformat(timespec='seconds'), 'step': None, 'step_started': None,
        'done': [], 'error': None, 'detail': ''}
def _first_line(cell):
    for l in cell.splitlines():
        l = l.strip()
        if l: return l.lstrip('#').strip()[:90]
    return '(empty cell)'
def _pre(info):
    PROG['step'] = _first_line(info.raw_cell); PROG['step_started'] = _dt.datetime.now().isoformat(timespec='seconds'); PROG['detail'] = ''
def _post(result):
    if result.error_in_exec is not None:
        PROG['error'] = {'step': PROG['step'], 'type': type(result.error_in_exec).__name__, 'message': str(result.error_in_exec)[:300]}
    else:
        PROG['done'].append(PROG['step'])
    PROG['step'] = None
ip = get_ipython()
for ev, fn in (('pre_run_cell', _pre), ('post_run_cell', _post)):
    try: ip.events.unregister(ev, fn)
    except Exception: pass
    ip.events.register(ev, fn)
def _beat():
    while True:
        try:
            PROG['updated'] = _dt.datetime.now().isoformat(timespec='seconds')
            tmp = OUT + '/progress.tmp'; _json.dump(PROG, open(tmp, 'w'), indent=1); os.replace(tmp, OUT + '/progress.json')
        except Exception: pass
        _time.sleep(20)
if not any(t.name == 'arcroad-progress' for t in threading.enumerate()):
    threading.Thread(target=_beat, name='arcroad-progress', daemon=True).start()
print('Live progress ->', OUT + '/progress.json (every 20 s)')


In [ ]:
# 3. Stops to verify. Default: Field Day 1 sample (24 flagged median gaps + 2 targets).
#    For a city export, set SOURCE to a CSV with columns id,a_lat,a_lon,b_lat,b_lon.
SOURCE = '/content/arcroad/data/field/field_day_1_stops.json'
if SOURCE.endswith('.json'):
    raw = json.load(open(SOURCE))['stops']
    stops = [{'id': s['id'], 'a': tuple(s['a']), 'b': tuple(s['b']), 'role': 'flagged'} for s in raw if s.get('b') and s['id'].startswith('S')]
else:
    import pandas as pd
    stops = [{'id': str(r.id), 'a': (r.a_lat, r.a_lon), 'b': (r.b_lat, r.b_lon), 'role': 'flagged'} for r in pd.read_csv(SOURCE).itertuples()]
print(len(stops), 'flagged stops')

## Evidence A: model route (OpenStreetMap, drive network)

Three legal networks, one per vehicle class, from OpenStreetMap access tags:
- **4-wheeler (car):** public roads, minus `motor_vehicle`/`motorcar=no`.
- **2-wheeler:** the car network plus ways open to motorcycles (`motorcycle=yes`), minus `motorcycle=no`.
- **Heavy (truck/bus):** the car network minus `hgv=no`, `maxweight` under 10 t and `maxheight` under 4 m.

One-way rules apply to all three. Limits: India's OSM rarely tags `oneway:motorcycle` or truck time bans, so the classes differ only where tags exist.

In [ ]:
import osmnx as ox, networkx as nx, random, re
from shapely.geometry import LineString as _LS0, Point as _Pt
ox.settings.log_console = False
ox.settings.useful_tags_way = list(set(ox.settings.useful_tags_way + ['access','motor_vehicle','motorcar','motorcycle','hgv','maxweight','maxheight','oneway']))
def bbox_of(stops, pad=0.03):
    lats = [p[0] for s in stops for p in (s['a'], s['b'])]; lons = [p[1] for s in stops for p in (s['a'], s['b'])]
    return (min(lons)-pad, min(lats)-pad, max(lons)+pad, max(lats)+pad)
W,S,E,N = bbox_of(stops)
CF = '["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified|residential|living_street|service|road|track|motorway_link|trunk_link|primary_link|secondary_link|tertiary_link"]'
G_all = ox.graph_from_bbox((W,S,E,N), custom_filter=CF, simplify=True, retain_all=True)
OSM_EXTRACT = dt.datetime.utcnow().isoformat(timespec='minutes') + 'Z (Overpass via OSMnx)'

def tag(d, k):
    v = d.get(k); v = v[0] if isinstance(v, list) else v
    return (v or '').lower()
def num(v):
    m = re.search(r'[0-9.]+', v or ''); return float(m.group()) if m else None
CAR_HW_EXCLUDE = {'track'}
def allowed(d, cls):
    if tag(d,'access') in ('no','private') and tag(d, {'car':'motorcar','two_wheeler':'motorcycle','heavy':'hgv'}[cls]) not in ('yes','designated'): return False
    if tag(d,'motor_vehicle') == 'no' and not (cls=='two_wheeler' and tag(d,'motorcycle')=='yes'): return False
    hw = tag(d,'highway')
    if cls == 'car':
        return tag(d,'motorcar') != 'no' and hw not in CAR_HW_EXCLUDE
    if cls == 'two_wheeler':
        if tag(d,'motorcycle') == 'no': return False
        return hw not in CAR_HW_EXCLUDE or tag(d,'motorcycle') == 'yes'
    if cls == 'heavy':
        if tag(d,'hgv') == 'no' or hw in CAR_HW_EXCLUDE or hw == 'living_street': return False
        w, h = num(tag(d,'maxweight')), num(tag(d,'maxheight'))
        return not ((w is not None and w < 10) or (h is not None and h < 4))
CLASSES = ('two_wheeler', 'car', 'heavy')
NET = {c: G_all.edge_subgraph([(u,v,k) for u,v,k,d in G_all.edges(keys=True, data=True) if allowed(d, c)]).copy() for c in CLASSES}
for c in CLASSES: print(c, NET[c].number_of_edges(), 'edges')

def _edge_geom(Gc, u, v, k):
    d = Gc.edges[u, v, k]
    return d.get('geometry') or _LS0([(Gc.nodes[u]['x'], Gc.nodes[u]['y']), (Gc.nodes[v]['x'], Gc.nodes[v]['y'])])
def _anchors(Gc, p):
    """Where a point sits on the network: the nearest edge (and its reverse, if two-way), with the fraction along it."""
    u, v, k = ox.distance.nearest_edges(Gc, p[1], p[0])
    out = []
    for (x, y) in ((u, v), (v, u)):
        if not Gc.has_edge(x, y): continue
        kk = min(Gc[x][y], key=lambda q: Gc.edges[x, y, q]['length'])
        geom = _edge_geom(Gc, x, y, kk); L = Gc.edges[x, y, kk]['length']
        f = geom.project(_Pt(p[1], p[0]), normalized=True)
        if (x, y) == (v, u) and Gc.edges[x, y, kk].get('geometry') is None: pass
        out.append((x, y, L, f))
    return out
def route_m(Gc, a, b):
    """Shortest legal distance from point a to point b, starting and ending mid-edge (not at the nearest junction)."""
    try:
        best = None
        for (ua, va, La, fa) in _anchors(Gc, a):
            for (ub, vb, Lb, fb) in _anchors(Gc, b):
                if (ua, va) == (ub, vb) and fb >= fa: cand = (fb - fa) * La
                else:
                    try: cand = (1 - fa) * La + nx.shortest_path_length(Gc, va, ub, weight='length') + fb * Lb
                    except nx.NetworkXNoPath: continue
                best = cand if best is None else min(best, cand)
        return None if best is None else round(float(best), 1)
    except Exception: return None

### Controls (Rule 5)
The sample has no unflagged stops, so controls are generated: for each flagged stop, two points 60-150 m apart **along one two-way road** near it. A real drive there should MATCH the model. Seeded, so reruns pick the same controls.

In [ ]:
from shapely.geometry import LineString as _LS
rng = random.Random(20261031)
def make_control(s, k):
    cands = []
    for u, v, kk, d in NET['car'].edges(keys=True, data=True):
        if d.get('oneway') or d.get('length', 0) < 160: continue
        if tag(d,'highway') not in ('primary','secondary','tertiary','residential','unclassified'): continue
        y, x = NET['car'].nodes[u]['y'], NET['car'].nodes[u]['x']
        if haversine_m(x, y, s['a'][1], s['a'][0]) < 2000: cands.append(d.get('geometry') or _LS([(x,y),(NET['car'].nodes[v]['x'], NET['car'].nodes[v]['y'])]))
    if not cands: return None
    g = rng.choice(cands); L = g.length; f0 = rng.uniform(0.05, 0.35)
    p0 = g.interpolate(f0, normalized=True); 
    for f1 in (f0 + 0.45, f0 + 0.35, f0 + 0.6):
        p1 = g.interpolate(min(f1, 0.95), normalized=True)
        dm = haversine_m(p0.x, p0.y, p1.x, p1.y)
        if 60 <= dm <= 150: break
    return {'id': f'C{k:02d}', 'a': (p0.y, p0.x), 'b': (p1.y, p1.x), 'role': 'control'}
controls = [c for c in (make_control(s, i+1) for i, s in enumerate(stops)) if c]
stops = [s for s in stops if s['role']=='flagged'] + controls
# Both directions of travel, each checked on its own: A->B and B->A can have different legal routes (one-way carriageways, U-turn places).
for s in stops: s['base'], s['dir'] = s['id'], 'A→B'
stops = stops + [dict(s, id=s['id'] + '_BA', a=s['b'], b=s['a'], dir='B→A') for s in stops]
for s in stops:
    s['straight_m'] = round(haversine_m(s['a'][1], s['a'][0], s['b'][1], s['b'][0]), 1)
    for c in CLASSES: s[f'net_{c}_m'] = route_m(NET[c], s['a'], s['b'])
    s['network_m'] = s['net_car_m']
print(len(controls), 'controls added; both directions:', sum(s['network_m'] is not None for s in stops), 'of', len(stops), 'routed for cars')

## Evidence B: real drives (public OpenStreetMap GPS traces)

In [ ]:
import requests, xml.etree.ElementTree as ET
UA = {'User-Agent': 'arcroad-verify/0.1 (civic research; contact via arcroad.org)'}
MAX_GAP_M = 60.0   # pre-registered density gate for untimed public traces

def _ekey(e): return frozenset((e[0], e[1]))
_SIDE_EDGE = {}
def side_check(s, track):
    """B-15: a sample only counts as 'at A' / 'at B' if it snaps to that side's own carriageway in the car graph."""
    G = NET['car']
    if s['id'] not in _SIDE_EDGE:
        ea = ox.distance.nearest_edges(G, s['a'][1], s['a'][0]); eb = ox.distance.nearest_edges(G, s['b'][1], s['b'][0])
        _SIDE_EDGE[s['id']] = {'A': _ekey(ea), 'B': _ekey(eb)}
    want = _SIDE_EDGE[s['id']]
    if not track: return None
    es = ox.distance.nearest_edges(G, [p[2] for p in track], [p[1] for p in track])
    snapped = {(p[1], p[2]): _ekey(e) for p, e in zip(track, es)}
    return lambda pt, side: snapped.get((pt[1], pt[2])) == want[side]

def trackpoints(lat, lon, half=0.0015, pages=6):
    """Segments of (time|None, lat, lon) around a point. Public traces arrive as ordered, untimed segments."""
    segs = []
    for page in range(pages):
        url = f'https://api.openstreetmap.org/api/0.6/trackpoints?bbox={lon-half},{lat-half},{lon+half},{lat+half}&page={page}'
        r = requests.get(url, headers=UA, timeout=30); time.sleep(1.0)
        if r.status_code != 200: break
        root = ET.fromstring(r.content); n = 0
        for seg in root.iter():
            if seg.tag.rsplit('}',1)[-1] != 'trkseg': continue
            pts = []
            for p in seg:
                if p.tag.rsplit('}',1)[-1] != 'trkpt': continue
                t = None
                for c in p:
                    if c.tag.rsplit('}',1)[-1] == 'time' and c.text:
                        t = dt.datetime.fromisoformat(c.text.strip().replace('Z','+00:00')).timestamp()
                pts.append((t, float(p.get('lat')), float(p.get('lon')))); n += 1
            if len(pts) >= gt.MIN_RUN * 2: segs.append(pts)
        if n < 5000: break
    return segs

def max_gap_m(track, i, j):
    return max((haversine_m(track[k][2], track[k][1], track[k+1][2], track[k+1][1]) for k in range(i, j)), default=0.0)

def evidence_B(s):
    best = None; passes = 0
    for seg in trackpoints((s['a'][0]+s['b'][0])/2, (s['a'][1]+s['b'][1])/2):
        for track in (seg, list(reversed(seg))):   # a public segment may run either way; reversing is fine for distance only if untimed
            if track is not seg and track[0][0] is not None: continue
            f = gt.find_pass(track, s['a'], s['b'], accept=side_check(s, track))
            if not f: continue
            i, j = f
            if max_gap_m(track, i, j) > MAX_GAP_M: continue
            passes += 1; L = gt.path_length_m(track, i, j)
            best = L if best is None else min(best, L)
    s['traces_passes'] = passes
    s['driven_m'] = None if best is None else round(best, 1)
    if best is None: s['B'] = 'NO_TRACE'
    elif s['network_m'] is None: s['B'] = 'BETWEEN'
    else: s['B'] = gt.verdict(best, s['network_m'])

for k, s in enumerate(stops):
    PROG['detail'] = f'GPS traces: stop {k+1} of {len(stops)}'; evidence_B(s); print(f"{s['id']:>4} {s['role']:<8} car {s['network_m']} 2w {s['net_two_wheeler_m']} heavy {s['net_heavy_m']} driven {s['driven_m']} -> {s['B']} ({s['traces_passes']} passes)")

### Power check (Rule 5)
If the controls do not mostly come back MATCH, Evidence B has no power here and is **not used**.

In [ ]:
ctrl = [s for s in stops if s['role']=='control']
ctrl_scored = [s for s in ctrl if s['B'] not in ('NO_TRACE',)]   # controls nobody drove past say nothing
ctrl_match = sum(s['B']=='MATCH' for s in ctrl_scored)
B_USABLE = len(ctrl_scored) >= 3 and ctrl_match / len(ctrl_scored) >= 0.6
print(f'controls with traces: {len(ctrl_scored)}/{len(ctrl)}, MATCH {ctrl_match}  ->  Evidence B usable: {B_USABLE}')

## Evidence C: aerial-imagery road detections (Microsoft, ODbL)

In [ ]:
# Reads only India's partition, once, and caches it in Drive (first run takes a while).
import duckdb, shapely
from shapely import wkb
from shapely.geometry import LineString, box
CACHE = OUT + '/ms_roads_bbox.parquet'
con = duckdb.connect(); con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial;")
if not os.path.exists(CACHE):
    src = "https://data.source.coop/nlebovits/microsoft-ml-road-detections/road-detections/by_country/country=IND/*.parquet"
    try:
        con.sql(f"""COPY (SELECT * FROM read_parquet('{src}') WHERE ST_Intersects(ST_GeomFromWKB(geometry), ST_MakeEnvelope({W},{S},{E},{N}))) TO '{CACHE}' (FORMAT parquet)""")
    except Exception as e:
        print('Could not read the mirror directly:', e, '\nCheck the partition name in collection.json (it is Microsoft\'s region code, not always ISO).')
ms = [wkb.loads(bytes(g)) for (g,) in con.sql(f"SELECT ST_AsWKB(ST_GeomFromWKB(geometry)) FROM read_parquet('{CACHE}')").fetchall()] if os.path.exists(CACHE) else []
if not ms: print('Microsoft imagery layer: no data this run (dispute-only source; results stay valid without it).')
tree = shapely.STRtree(ms) if ms else None
print(len(ms), 'detected road segments in the study box')
def evidence_C(s):
    if tree is None: s['C'] = 'NO_DATA'; return
    ab = LineString([(s['a'][1], s['a'][0]), (s['b'][1], s['b'][0])])
    hits = [ms[i] for i in tree.query(ab) if ms[i].crosses(ab)]
    s['C_crossings'] = len(hits)
    s['C'] = 'IMAGERY_SHOWS_LINK' if hits else 'NO_LINK_SEEN'
for s in stops: evidence_C(s)
print({v: sum(s['C']==v for s in stops) for v in ('IMAGERY_SHOWS_LINK','NO_LINK_SEEN','NO_DATA')})

## Evidence B2: street-level photos with timestamps (Mapillary, CC BY-SA)
Mapillary sequences are photos taken every second or two from a moving vehicle, each with a capture time. A sequence that goes from side A to side B is a **timed drive**, so it passes the full `groundtruth` gate (including the 15 s gap rule), and the photos at the spot can be opened to see the median for yourself.
Needs a free Mapillary access token (mapillary.com → Developers → register an app) saved in Colab **Secrets** as `MAPILLARY_TOKEN`. Only image IDs are kept, never the photos.

In [ ]:
USE_MAPILLARY = bool(MAPILLARY_TOKEN)
META_MLY = {}
REJECTED_MLY = []   # v05: passes thrown out by the sanity rules, kept for the record
def mly_track(s, half=0.002):
    tok = MAPILLARY_TOKEN
    lat, lon = (s['a'][0]+s['b'][0])/2, (s['a'][1]+s['b'][1])/2
    url = ('https://graph.mapillary.com/images?fields=id,captured_at,geometry,sequence,compass_angle,make,model,is_pano'
           f'&bbox={lon-half},{lat-half},{lon+half},{lat+half}&limit=2000')
    r = requests.get(url, headers={'Authorization': f'OAuth {tok}'}, timeout=60); time.sleep(0.5)
    if r.status_code != 200: return None
    seqs = {}
    for im in r.json().get('data', []):
        x, y = im['geometry']['coordinates']; seqs.setdefault(im['sequence'], []).append((im['captured_at']/1000.0, y, x, im['id'], im.get('compass_angle')))
        META_MLY.setdefault(s['id'], []).append({k: im.get(k) for k in ('captured_at','compass_angle','make','model','is_pano')})
    return seqs
def evidence_B2(s):
    s['B2'] = 'OFF' if not USE_MAPILLARY else 'NO_PHOTOS'; s['B2_photos'] = ''
    if not USE_MAPILLARY: return
    seqs = mly_track(s)
    if seqs is None: s['B2'] = 'API_ERROR'; return
    best = None
    for pts in seqs.values():
        pts.sort(); track = [(t, y, x) for t, y, x, _, _ in pts]
        radius = min(gt.VISIT_RADIUS_M, 0.45 * s['straight_m'])   # v05: never wider than half the A-B gap
        f = gt.find_pass(track, s['a'], s['b'], radius=radius, accept=side_check(s, track))
        if not f: continue
        i, j = f; gap = gt.max_gap_s(track, i, j)
        if gap is None or gap > gt.MAX_SAMPLE_GAP_S: continue
        L = gt.path_length_m(track, i, j)
        if L < s['straight_m']: REJECTED_MLY.append((s['id'], 'shorter than straight')); continue   # v05: physically impossible as a crossing
        ha, hb = pts[i][4], pts[j][4]
        if ha is not None and hb is not None:
            turn = abs((hb - ha + 180) % 360 - 180)
            if turn < 120: REJECTED_MLY.append((s['id'], 'no heading reversal')); continue       # v05: a crossing needs a U-turn
        if best is None or L < best[0]: best = (L, [pts[k][3] for k in (i, (i+j)//2, j)])
    if best:
        s['B2_driven_m'] = round(best[0], 1)
        s['B2'] = gt.verdict(best[0], s['network_m']) if s['network_m'] else 'BETWEEN'
        s['B2_photos'] = ' '.join(f'https://www.mapillary.com/app/?pKey={k}' for k in best[1])
for _k, s in enumerate(stops): PROG['detail'] = f'Mapillary: stop {_k+1} of {len(stops)}'; evidence_B2(s)
ctrl2 = [s for s in stops if s['role']=='control' and s['B2'] not in ('NO_PHOTOS','OFF','API_ERROR')]
B2_USABLE = len(ctrl2) >= 3 and sum(s['B2']=='MATCH' for s in ctrl2) / len(ctrl2) >= 0.6
print('rejected by v05 sanity rules:', len(REJECTED_MLY))
print({v: sum(s['B2']==v for s in stops) for v in set(s['B2'] for s in stops)}, '| controls scored', len(ctrl2), '-> B2 usable:', B2_USABLE)

## Evidence C2: Overture Maps roads not taken from OpenStreetMap
Overture (Linux Foundation; Amazon, Meta, Microsoft, TomTom) publishes road segments with a `sources` field. Segments whose source is **not** OpenStreetMap (e.g. TomTom-contributed) are an independent map. One that crosses the A–B line suggests a link OSM is missing. Like imagery, this can only dispute.

In [ ]:
OVERTURE_RELEASE = None   # e.g. '2026-10-22.0'; None = latest from the STAC catalog
def latest_release():
    try:
        cat = requests.get('https://stac.overturemaps.org/catalog.json', timeout=30).json()
        rel = sorted(l['href'].split('/')[-2] for l in cat['links'] if l.get('rel') == 'child')
        return rel[-1]
    except Exception as e:
        print('STAC lookup failed; set OVERTURE_RELEASE by hand (docs.overturemaps.org release notes).', e); return None
OVERTURE_RELEASE = OVERTURE_RELEASE or latest_release()
OCACHE = OUT + f'/overture_{OVERTURE_RELEASE}_bbox.parquet'
import glob as _glob
_have = sorted(_glob.glob(OUT + '/overture_*_bbox.parquet'))
if _have and not os.path.exists(OCACHE): OCACHE = _have[-1]; print('Reusing saved Overture extract', os.path.basename(OCACHE))
if OVERTURE_RELEASE and not os.path.exists(OCACHE):
    con.sql("SET s3_region='us-west-2';")
    con.sql(f"""COPY (SELECT id, sources, class, geometry FROM read_parquet('s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=transportation/type=segment/*', hive_partitioning=1)
        WHERE subtype='road' AND bbox.xmin < {E} AND bbox.xmax > {W} AND bbox.ymin < {N} AND bbox.ymax > {S}) TO '{OCACHE}' (FORMAT parquet)""")
ov = []
if os.path.exists(OCACHE):
    for gid, src, cls, g in con.sql(f"SELECT id, sources, class, ST_AsWKB(geometry) FROM read_parquet('{OCACHE}')").fetchall():
        names = {str(x.get('dataset','')).lower() for x in (src or []) if isinstance(x, dict)}
        if names and not any('openstreetmap' in n for n in names): ov.append(wkb.loads(bytes(g)))
otree = shapely.STRtree(ov) if ov else None
print(len(ov), 'non-OSM Overture road segments in the box (release', OVERTURE_RELEASE, ')')
def evidence_C2(s):
    if otree is None: s['C2'] = 'NO_DATA'; return
    ab = LineString([(s['a'][1], s['a'][0]), (s['b'][1], s['b'][0])])
    s['C2'] = 'OTHER_MAP_SHOWS_LINK' if any(ov[i].crosses(ab) for i in otree.query(ab)) else 'NO_LINK_SEEN'
for s in stops: evidence_C2(s)

## Evidence E: U-turn signs seen in street photos (Mapillary detections, CC BY-SA)
Mapillary detects traffic signs in its photos. A "U-turn allowed" sign within 60 m of either side means a legal crossing may exist that the map lacks (a dispute). A "no U-turn" sign fits the detour. Signs only move a level through `pipeline/evidence.py` (U_TURN_SIGN disputes; NO_U_TURN_SIGN is noted).

In [ ]:
SIGN_RADIUS_M = 60.0   # pre-registered, 2 Nov 2026
def evidence_E(s, half=0.0012):
    s['E'] = 'OFF' if not MAPILLARY_TOKEN else 'NO_SIGNS'
    if not MAPILLARY_TOKEN: return
    lat, lon = s['a']
    url = (f'https://graph.mapillary.com/map_features?fields=id,object_value,geometry,last_seen_at'
           f'&bbox={lon-half},{lat-half},{lon+half},{lat+half}&limit=500')
    try:
        r = requests.get(url, headers={'Authorization': f'OAuth {MAPILLARY_TOKEN}'}, timeout=60); time.sleep(0.5)
        feats = r.json().get('data', []) if r.status_code == 200 else None
    except Exception:
        feats = None
    if feats is None: s['E'] = 'API_ERROR'; return
    near = []
    for f_ in feats:
        v = (f_.get('object_value') or '').lower()
        if 'u-turn' not in v and 'u_turn' not in v: continue
        x, y = f_['geometry']['coordinates']
        d = min(haversine_m(x, y, p[1], p[0]) for p in (s['a'], s['b']))
        if d <= SIGN_RADIUS_M: near.append(('NO_U_TURN_SIGN' if ('no-u-turn' in v or 'no_u_turn' in v) else 'U_TURN_SIGN', round(d), f_.get('last_seen_at')))
    if near:
        s['E'] = 'U_TURN_SIGN' if any(k == 'U_TURN_SIGN' for k, _, _ in near) else 'NO_U_TURN_SIGN'
        s['E_detail'] = near
for _k, s in enumerate(stops): PROG['detail'] = f'Signs: stop {_k+1} of {len(stops)}'; evidence_E(s)
print({v: sum(s['E']==v for s in stops) for v in set(s['E'] for s in stops)})

## Evidence F: turn restrictions mapped in OpenStreetMap
Restriction relations (`no_u_turn`, `only_straight_on`) at the junctions near each place. Context for the detour, never a level change on their own.

In [ ]:
OVERPASS = 'https://overpass-api.de/api/interpreter'
q = f'[out:json][timeout:120];relation["type"="restriction"]["restriction"~"no_u_turn|only_straight_on"]({S},{W},{N},{E});out body;>;out skel qt;'
try:
    ov = requests.post(OVERPASS, data={'data': q}, headers=UA, timeout=180).json()
except Exception as ex:
    ov = {'elements': []}; print('Overpass failed:', type(ex).__name__)
nodes = {e['id']: (e['lat'], e['lon']) for e in ov['elements'] if e['type'] == 'node'}
RESTR = []
for e in ov['elements']:
    if e['type'] != 'relation': continue
    via = [m for m in e.get('members', []) if m.get('role') == 'via' and m.get('type') == 'node' and m['ref'] in nodes]
    if via: RESTR.append((e['tags'].get('restriction'), nodes[via[0]['ref']]))
for s in stops:
    hits = [r for r, p in RESTR if min(haversine_m(p[1], p[0], q_[1], q_[0]) for q_ in (s['a'], s['b'])) <= 150]
    s['F'] = ','.join(sorted(set(hits))) or 'NONE'
print(len(RESTR), 'restrictions in the box;', sum(s['F'] != 'NONE' for s in stops), 'places have one within 150 m')

## Own drives and road class
arcroad's own drive results (published per pass on arcroad.org/field) are read in, so a place driven by arcroad gets the same level here as on the website. Road class marks access-controlled roads (motorway, or a trunk road named Outer Ring Road), where a detour to an interchange is by design.

In [ ]:
import glob as _g
OWN = {}
for fn in sorted(_g.glob('/content/arcroad/data/field/*-results.json')):
    for p in json.load(open(fn))['passes']:
        if p.get('level') == 1:
            OWN[(p['id'], {'ab': 'A→B', 'ba': 'B→A'}[p['dir']])] = p
for s in stops:
    o = OWN.get((s.get('base'), s.get('dir')))
    s['own'] = o['verdict'] if o else 'NOT_DRIVEN'
    s['own_driven_m'] = o['driven_m'] if o else None
    t = way_tags_at(NET['car'], s['a'])
    hw, nm = str(t.get('highway', '')), str(t.get('name', '')) + ' ' + str(t.get('ref', ''))
    s['road_class'] = 'access-controlled' if ('motorway' in hw or 'outer ring' in nm.lower() or 'orr' in nm.lower().split()) else 'city road'
print(sum(s['own'] != 'NOT_DRIVEN' for s in stops), 'directions driven by arcroad;',
      sum(s['road_class'] == 'access-controlled' for s in stops), 'directions on access-controlled roads')

## Optional: KartaView photo runs (off)
KartaView (open street-level imagery) could be a second drive source. Leave `USE_KARTAVIEW = False` until its current API and licence terms are checked and written into docs/ENGINE.md.

In [ ]:
USE_KARTAVIEW = False   # terms not yet checked (2 Nov 2026)
for s in stops: s['B3'] = 'OFF' 

## Evidence D: TomTom route (off by default; private check only)
Read TomTom's developer terms first (sign in at developer.tomtom.com). Private diagnostic only (constitution Art 4c): its verdict is kept in your CSV for your eyes, never in the site file and never in a published level. Only a verdict word is kept, never TomTom's route or geometry. Runs once per vehicle class (motorcycle, car, truck), so 3 requests per stop. Free tier: 2,500 a day, about 800 stops. Key goes in Colab **Secrets** as `TOMTOM_KEY`, never in a cell.

In [ ]:
USE_TOMTOM = False
def evidence_D(s):
    if not USE_TOMTOM: s['D'] = 'OFF'; return
    key = TOMTOM_KEY
    for c, mode in (('two_wheeler','motorcycle'), ('car','car'), ('heavy','truck')):
        url = f"https://api.tomtom.com/routing/1/calculateRoute/{s['a'][0]},{s['a'][1]}:{s['b'][0]},{s['b'][1]}/json?routeType=shortest&travelMode={mode}&key={key}"
        r = requests.get(url, timeout=30); time.sleep(0.3)
        try: L = r.json()['routes'][0]['summary']['lengthInMeters']
        except Exception: s[f'D_{c}'] = 'NO_ROUTE'; continue
        net = s.get(f'net_{c}_m')
        s[f'D_{c}'] = gt.verdict(L, net) if net else 'BETWEEN'   # L itself is discarded
    s['D'] = s.get('D_car', 'OFF')
for s in stops: evidence_D(s)

## Metadata for triangulation
Everything public that bears on each stop, kept per stop (no usernames, no raw trace coordinates):
- **OSM tags** of the roads under A and B (type, one-way, lanes, divider, name/ref, way ids) for each vehicle class.
- **OSM edit history**: when those ways were last edited and how many versions they have (a recent edit can mean a new median opening or a fix).
- **OSM Notes** within 150 m: open public reports of map problems (count, ids, dates).
- **Mapillary**: number of sequences, oldest and newest capture date (recency matters: medians get cut and closed), camera types, headings.
- **Agreement count**: how many independent sources support or dispute the modelled detour.

In [ ]:
# v05.1 (2 Nov 2026): the old version called osmnx nearest_edges for every stop and class, rebuilding a city-wide
# edge index each time; on Colab that froze the run (even the progress file stopped). Now one index per class, built once.
from shapely.geometry import Point as _Pt, LineString as _LS2
from shapely import STRtree as _STR
_EIDX = {}
def _edge_index(c):
    if c not in _EIDX:
        G = NET[c]; keys, geoms = [], []
        for u, v, k, d in G.edges(keys=True, data=True):
            g = d.get('geometry') or _LS2([(G.nodes[u]['x'], G.nodes[u]['y']), (G.nodes[v]['x'], G.nodes[v]['y'])])
            keys.append((u, v, k)); geoms.append(g)
        _EIDX[c] = (_STR(geoms), keys, G)
        print(f'  edge index for {c}: {len(keys):,} edges')
    return _EIDX[c]
def way_tags_at(c, p):
    try:
        tree, keys, G = _edge_index(c); u, v, k = keys[int(tree.nearest(_Pt(p[1], p[0])))]; d = G.edges[u, v, k]
        keep = ('osmid','highway','oneway','lanes','name','ref','divider','maxspeed','junction','access','motorcycle','hgv','maxweight','maxheight','bridge','tunnel')
        return {k2: d.get(k2) for k2 in keep if d.get(k2) is not None}
    except Exception: return {}
META_BUDGET_S = 20 * 60   # online lookups (edit history, notes) stop after 20 minutes; the rest of the run still completes
_META_T0 = time.time()
def _budget_left(): return time.time() - _META_T0 < META_BUDGET_S
def osm_way_history(way_ids):
    out = {}
    for w in list(way_ids)[:6]:
        if not _budget_left(): break
        try:
            r = requests.get(f'https://api.openstreetmap.org/api/0.6/way/{w}/history.json', headers=UA, timeout=(5, 15)); time.sleep(1.0)
            els = r.json()['elements']; out[str(w)] = {'versions': len(els), 'last_edit': els[-1]['timestamp'], 'first_seen': els[0]['timestamp']}
        except Exception: pass
    return out
def osm_notes(s, half=0.0014):
    lat, lon = s['a']
    if not _budget_left(): return {'count': None, 'skipped': 'time budget'}
    try:
        r = requests.get(f'https://api.openstreetmap.org/api/0.6/notes.json?bbox={lon-half},{lat-half},{lon+half},{lat+half}&closed=30', headers=UA, timeout=(5, 15)); time.sleep(1.0)
        fs = r.json().get('features', [])
        return {'count': len(fs), 'open': sum(f['properties']['status']=='open' for f in fs),
                'ids': [f['properties']['id'] for f in fs], 'dates': [f['properties']['date_created'][:10] for f in fs]}
    except Exception: return {'count': None}
def flat_ids(x): return [i for i in (x if isinstance(x, list) else [x]) if i is not None]
EVID = {}
for _k, s in enumerate(stops):
    PROG['detail'] = f'Metadata: stop {_k+1} of {len(stops)}'
    e = {'tags_A': {c: way_tags_at(c, s['a']) for c in CLASSES}, 'tags_B': {c: way_tags_at(c, s['b']) for c in CLASSES}}
    ways = set(flat_ids(e['tags_A']['car'].get('osmid')) + flat_ids(e['tags_B']['car'].get('osmid')))
    e['osm_history'] = osm_way_history(ways)
    e['osm_notes'] = osm_notes(s)
    m = META_MLY.get(s['id'], [])
    if m:
        ts = sorted(x['captured_at'] for x in m if x.get('captured_at'))
        e['mapillary'] = {'images': len(m), 'oldest': dt.datetime.utcfromtimestamp(ts[0]/1000).date().isoformat(),
                          'newest': dt.datetime.utcfromtimestamp(ts[-1]/1000).date().isoformat(),
                          'cameras': sorted({f"{x.get('make') or '?'} {x.get('model') or ''}".strip() for x in m})[:8],
                          'panoramas': sum(bool(x.get('is_pano')) for x in m)}
    support = [k for k, ok in (('osm_traces', B_USABLE and s['B']=='MATCH'), ('mapillary', B2_USABLE and s.get('B2')=='MATCH')) if ok]
    dispute = [k for k, ok in (('osm_traces', B_USABLE and s['B']=='SHORTER'), ('mapillary', B2_USABLE and s.get('B2')=='SHORTER'),
                               ('ms_imagery', s['C']=='IMAGERY_SHOWS_LINK'), ('overture_non_osm', s.get('C2')=='OTHER_MAP_SHOWS_LINK')) if ok]
    e['support'], e['dispute'] = support, dispute
    s['n_support'], s['n_dispute'] = len(support), len(dispute)
    last = [h['last_edit'] for h in e['osm_history'].values()]
    s['osm_last_edit'] = max(last)[:10] if last else None
    s['osm_notes'] = e['osm_notes'].get('count')
    s['mly_newest'] = e.get('mapillary', {}).get('newest')
    s['divider_tag'] = e['tags_A']['car'].get('divider') or e['tags_B']['car'].get('divider')
    EVID[s['id']] = e
print('metadata captured for', len(EVID), 'stops')

## Combine into a published tier
- **Independently corroborated**: a usable drive source by other people (B or B2) says MATCH and no map disputes it. The modelled detour is real.
- **Conflicting: look at photos**: a drive says MATCH but another map shows a link. Open the Mapillary links.
- **Map-disputed**: B says SHORTER, or imagery/TomTom suggests a link. Do not publish the value; fix the map.
- **Unverified**: no usable evidence yet. Stays a candidate.

Public GPS traces carry no vehicle type and are mostly untimed, so Evidence B is compared with the **car** network. The 2-wheeler and heavy columns are model values per class; TomTom (if on) checks each class separately.

In [ ]:
EVID = globals().get('EVID', {})   # v05.1: never fail here if the metadata step was skipped
def tier(s):
    if s.get('own') == 'MATCH': return 'Driven by arcroad'
    if s.get('own') == 'SHORTER': return 'Driven by arcroad (map out of date)'
    disputes = s['C'] == 'IMAGERY_SHOWS_LINK' or s.get('C2') == 'OTHER_MAP_SHOWS_LINK' or s.get('E') == 'U_TURN_SIGN'   # TomTom (D) never moves a published level
    drove_short = (B_USABLE and s['B'] == 'SHORTER') or (B2_USABLE and s.get('B2') == 'SHORTER')
    drove_match = (B_USABLE and s['B'] == 'MATCH') or (B2_USABLE and s.get('B2') == 'MATCH')
    if drove_short: return 'Map-disputed'
    if drove_match and not disputes: return 'Independently corroborated'
    if drove_match and disputes: return 'Conflicting: look at photos'
    if disputes: return 'Map-disputed'
    return 'Unverified'
import pandas as pd
for s in stops:
    s['tier'] = tier(s)
    for c in CLASSES:   # how much worse each class has it than a car, on the model
        n = s.get(f'net_{c}_m'); s[f'detour_{c}_m'] = None if n is None else round(n - s['straight_m'], 1)
cols = ['id','base','dir','role','straight_m'] + [f'net_{c}_m' for c in CLASSES] + [f'detour_{c}_m' for c in CLASSES] + ['driven_m','traces_passes','B','B2','B2_driven_m','B2_photos','C','C_crossings','C2','D','E','F','own','own_driven_m','road_class'] + [f'D_{c}' for c in CLASSES] + ['n_support','n_dispute','osm_last_edit','osm_notes','mly_newest','divider_tag','tier']
res = pd.DataFrame([{c: s.get(c) for c in cols} for s in stops])
meta = {'run': RUN_DATE, 'commit': COMMIT, 'osm': OSM_EXTRACT, 'source': SOURCE, 'B_usable': B_USABLE, 'B2_usable': B2_USABLE, 'overture_release': OVERTURE_RELEASE,
        'classes': list(CLASSES), 'gate': {'MATCH_ABS_M': gt.MATCH_ABS_M, 'MATCH_REL': gt.MATCH_REL, 'SHORTER_REL': gt.SHORTER_REL, 'MAX_GAP_M': MAX_GAP_M},
        'attribution': ['© OpenStreetMap contributors, ODbL', 'Contains road detections from Microsoft, mined from Bing Maps imagery, licensed under ODbL', 'Street-level imagery © Mapillary contributors, CC BY-SA 4.0', 'Overture Maps Foundation data, per-source licences (overturemaps.org/attribution)']}
stamp = dt.datetime.now().strftime('%Y%m%d_%H%M')
res.to_csv(f'{OUT}/verify_{stamp}.csv', index=False); import platform, importlib.metadata as im_
meta['environment'] = {'python': platform.python_version(), **{p: im_.version(p) for p in ('osmnx','networkx','shapely','duckdb')}}
meta['study_box'] = [W, S, E, N]
json.dump(meta, open(f'{OUT}/verify_{stamp}.meta.json','w'), indent=1)
json.dump(EVID, open(f'{OUT}/verify_{stamp}.evidence.json','w'), indent=1, default=str)
print(res['tier'].value_counts().to_string()); res

In [ ]:
# Routes for the website's animated detour: the model's car route for each flagged stop and direction.
# Illustration only: node-to-node shortest path on the car graph, so the ends sit on the nearest junctions.
feats = []
for s in [x for x in stops if x['role'] == 'flagged']:
    try:
        G = NET['car']
        na = ox.distance.nearest_nodes(G, s['a'][1], s['a'][0]); nb_ = ox.distance.nearest_nodes(G, s['b'][1], s['b'][0])
        path = nx.shortest_path(G, na, nb_, weight='length')
        coords = [[round(G.nodes[n]['x'], 6), round(G.nodes[n]['y'], 6)] for n in path]
        feats.append({'type': 'Feature', 'properties': {'id': s['base'], 'dir': s['dir'], 'model_m': s['network_m'], 'level': 'candidate'},
                      'geometry': {'type': 'LineString', 'coordinates': [[s['a'][1], s['a'][0]]] + coords + [[s['b'][1], s['b'][0]]]}})
    except Exception as e:
        print('no route drawn for', s['id'], type(e).__name__)
json.dump({'type': 'FeatureCollection', 'note': 'Model car routes, candidates, illustration only. OpenStreetMap contributors, ODbL.', 'features': feats},
          open(f"{OUT}/routes_{dt.datetime.now().strftime('%Y%m%d_%H%M')}.geojson", 'w'))
print(len(feats), 'routes saved for the website')


## Website export: accuracy per direction, per vehicle class
`site_evidence.json` is what the arcroad site reads to show, for every spot and each direction of travel, **how sure we are and why**:
- the modelled detour for 2-wheelers, cars and heavy vehicles;
- each evidence source's verdict, with its measured hit rate on the controls (how often it reproduced a known no-detour case), so accuracy is a measured number, not a claim;
- the evidence level (Driven-verified / Conflicting / Map-disputed / Unverified), dates of the newest evidence, and attribution.
Untimed public GPS traces cannot tell direction, so that source is marked *direction-agnostic*. Nothing goes live until Rohit says go.

In [ ]:
EVID = globals().get('EVID', {})   # v05.1: never fail here if the metadata step was skipped
def hit_rate(key, ok_vals=('MATCH',), skip=('NO_TRACE','NO_PHOTOS','OFF','API_ERROR','NO_DATA')):
    c = [s for s in stops if s['role']=='control' and s.get(key) not in skip and s.get(key) is not None]
    return {'controls_scored': len(c), 'reproduced': sum(s.get(key) in ok_vals for s in c),
            'rate': round(sum(s.get(key) in ok_vals for s in c) / len(c), 2) if c else None}
def clean_rate(key, bad):
    c = [s for s in stops if s['role']=='control' and s.get(key) not in ('NO_DATA', None)]
    return {'controls_checked': len(c), 'false_alarms': sum(s.get(key)==bad for s in c),
            'rate_clean': round(1 - sum(s.get(key)==bad for s in c) / len(c), 2) if c else None}
SOURCES = {
    'osm_traces': {'label': 'Public GPS traces (OpenStreetMap)', 'kind': 'drive', 'direction_aware': False, 'licence': 'ODbL', **hit_rate('B')},
    'mapillary':  {'label': 'Timed street-level photo runs (Mapillary)', 'kind': 'drive', 'direction_aware': True, 'licence': 'CC BY-SA 4.0', **hit_rate('B2')},
    'ms_imagery': {'label': 'Roads detected in aerial imagery (Microsoft)', 'kind': 'dispute-only', 'direction_aware': False, 'licence': 'ODbL', **clean_rate('C','IMAGERY_SHOWS_LINK')},
    'overture':   {'label': 'Roads from non-OSM sources (Overture Maps)', 'kind': 'dispute-only', 'direction_aware': False, 'licence': 'per source', **clean_rate('C2','OTHER_MAP_SHOWS_LINK')},
}
site = {'generated': RUN_DATE, 'commit': COMMIT, 'osm_extract': OSM_EXTRACT, 'gate': meta['gate'], 'sources': SOURCES,
        'attribution': meta['attribution'], 'spots': []}
for base in sorted({s['base'] for s in stops if s['role']=='flagged'}):
    spot = {'id': base, 'directions': []}
    for s in [x for x in stops if x['base']==base]:
        e = EVID.get(s['id'], {})
        spot['directions'].append({
            'dir': s['dir'], 'from': [round(v, 6) for v in s['a']], 'to': [round(v, 6) for v in s['b']],
            'straight_m': s['straight_m'],
            'detour_m': {c: s.get(f'detour_{c}_m') for c in CLASSES},
            'evidence': {'own_drive': s.get('own'), 'osm_traces': s['B'], 'mapillary': s.get('B2'), 'ms_imagery': s['C'], 'overture': s.get('C2'), 'signs': s.get('E'), 'osm_restrictions': s.get('F')},
            'road_class': s.get('road_class'),
            'support': e.get('support', []), 'dispute': e.get('dispute', []),
            'level': s['tier'],
            'observed': {'newest_photo_run': s.get('mly_newest'), 'osm_last_edit': s.get('osm_last_edit')},   # when the world was seen
            'ingested': RUN_DATE,                                                                         # when we checked
            'coverage': {'osm_trace_passes': s.get('traces_passes', 0), 'photo_images_near': e.get('mapillary', {}).get('images', 0), 'osm_notes_near': s.get('osm_notes')},
            'assumptions': 'Legal turns and one-way rules as tagged in OSM; vehicle access tags; no time-of-day bans modelled',
            'not_a_finding_of': 'safety hazard or legal restriction: a detour is a distance',
            'photos': s.get('B2_photos', '').split() if s.get('B2_photos') else [],
        })
    site['spots'].append(spot)
json.dump(site, open(f'{OUT}/site_evidence_{stamp}.json', 'w'), indent=1, default=str)
print(json.dumps(SOURCES, indent=1, default=str)); print(len(site['spots']), 'spots x 2 directions exported')

## Look for yourself: aerial view of disputed spots (viewing only, nothing saved)

In [ ]:
import folium
dis = [s for s in stops if s['tier'] != 'Independently corroborated']
m = folium.Map(location=dis[0]['a'] if dis else stops[0]['a'], zoom_start=18, tiles=None)
folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Esri World Imagery (view only)', name='Imagery').add_to(m)
for s in dis:
    folium.PolyLine([s['a'], s['b']], color='orange', tooltip=f"{s['id']}: {s['tier']}").add_to(m)
    folium.CircleMarker(s['a'], radius=4, color='white', fill=True, tooltip=f"{s['id']} A").add_to(m)
    folium.CircleMarker(s['b'], radius=4, color='orange', fill=True, tooltip=f"{s['id']} B").add_to(m)
m

## Scaling to every city
1. Run the arcroad pipeline export for a city (it now keeps both endpoints, B-09) and set `SOURCE` to that CSV.
2. OSM's trackpoints API is volunteer-run: keep the 1 request per second pause. A city of 10,000 candidates takes a few hours; run it in Colab Pro with the browser left open, or in batches.
3. Microsoft detections: the cache is per study box; delete `ms_roads_bbox.parquet` when the box changes.
4. Citizen confirmations (the "millions across the country" layer) live in a **separate labelled layer**, never mixed into published figures (AGENTS.md).
